In [17]:
import pandas as pd
import gc

df = pd.read_csv("../ex04/fines.csv")


In [52]:
%%timeit


def calculate_iloc(dataframe):
    result = []

    fines_idx = dataframe.columns.get_loc("Fines")
    refund_idx = dataframe.columns.get_loc("Refund")
    year_idx = dataframe.columns.get_loc("Year")

    for i in range(0, len(dataframe)):
        val = (
            dataframe.iloc[i, fines_idx]
            / dataframe.iloc[i, refund_idx]
            * dataframe.iloc[i, year_idx]
        )
        result.append(val)

    return result


df["Calculation"] = calculate_iloc(df)

25.3 ms ± 1.09 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [30]:
%%timeit


def calculate_iloc_v2(data):
    result = []
    for i in range(0, len(data)):
        val = data.iloc[i]["Fines"] / data.iloc[i]["Refund"] * data.iloc[i]["Year"]
        result.append(val)
    return result


df["Calculation"] = calculate_iloc_v2(df)

69.3 ms ± 2.61 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [31]:
%%timeit


def calculate_iterrows(dataframe):
    result = []
    for _, row in dataframe.iterrows():
        val = row["Fines"] / row["Refund"] * row["Year"]
        result.append(val)
    return result


df["Calculation"] = calculate_iterrows(df)

17.9 ms ± 634 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [32]:
%%timeit
df["Calculation"] = df.apply(lambda r: r["Fines"] / r["Refund"] * r["Year"], axis=1)


5.14 ms ± 304 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [33]:
%%timeit
df["Calculation"] = df["Fines"] / df["Refund"] * df["Year"]

92.4 μs ± 1.43 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [18]:
%%timeit
df["Calculation"] = df["Fines"].values / df["Refund"].values * df["Year"].values

33.2 μs ± 960 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [4]:
%%timeit
df[df["CarNumber"] == "O136HO197RUS"]

392 μs ± 138 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [19]:
df.set_index("CarNumber", inplace=True)

In [20]:
%%timeit
df.loc["O136HO197RUS"]

26.1 μs ± 654 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [22]:
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
Index: 930 entries, Y163O8161RUS to E555EE178RUS
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Refund       930 non-null    int64  
 1   Fines        930 non-null    float64
 2   Make         930 non-null    str    
 3   Model        919 non-null    str    
 4   Year         930 non-null    int64  
 5   Calculation  930 non-null    float64
dtypes: float64(2), int64(2), str(2)
memory usage: 214.3 KB


In [23]:
optimized_df = df.copy()
optimized_df.reset_index(inplace=True)

In [ ]:
float_cols = optimized_df.select_dtypes(include=["float64"]).columns
for col in float_cols:
    optimized_df[col] = optimized_df[col].astype("float32")

int_cols = optimized_df.select_dtypes(include=["int64"]).columns
for col in int_cols:
    optimized_df[col] = pd.to_numeric(optimized_df[col], downcast="integer")

optimized_df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   CarNumber    930 non-null    str    
 1   Refund       930 non-null    int8   
 2   Fines        930 non-null    float32
 3   Make         930 non-null    str    
 4   Model        919 non-null    str    
 5   Year         930 non-null    int16  
 6   Calculation  930 non-null    float32
dtypes: float32(2), int16(1), int8(1), str(3)
memory usage: 163.1 KB


In [ ]:
object_cols = optimized_df.select_dtypes(include=["object"]).columns
for col in object_cols:
    optimized_df[col] = optimized_df[col].astype("category")

optimized_df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   CarNumber    930 non-null    category
 1   Refund       930 non-null    int8    
 2   Fines        930 non-null    float32 
 3   Make         930 non-null    category
 4   Model        919 non-null    category
 5   Year         930 non-null    int16   
 6   Calculation  930 non-null    float32 
dtypes: category(3), float32(2), int16(1), int8(1)
memory usage: 46.5 KB


/tmp/ipykernel_53624/967819196.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = optimized_df.select_dtypes(include=['object']).columns


In [ ]:
%reset_selective -f ^df$

gc.collect()

6